# 01 — Data Ingestion & Data Lake Setup

**Objective**: Load the raw Home Credit Default Risk CSVs (bronze layer), inspect schemas and data quality, clean dtypes, and save as Parquet files (silver layer).

## Data Lake Architecture

| Layer | Location | Format | Description |
|-------|----------|--------|-------------|
| **Bronze** | `data/raw/` | CSV | Raw files as downloaded from Kaggle |
| **Silver** | `data/processed/` | Parquet | Cleaned, properly typed, validated |
| **Gold** | `data/features/` | Parquet | Feature-engineered, ready for training |

## Dataset Tables

| Table | Description |
|-------|-------------|
| `application_train/test` | Main application data (target is here) |
| `bureau` | Client's previous credits from other institutions |
| `bureau_balance` | Monthly balances of bureau credits |
| `previous_application` | Previous Home Credit applications |
| `POS_CASH_balance` | Monthly POS/cash loan snapshots |
| `credit_card_balance` | Monthly credit card snapshots |
| `installments_payments` | Repayment history |

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# Project paths
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"        # Bronze layer
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"  # Silver layer

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw data: {RAW_DIR}")
print(f"Processed data: {PROCESSED_DIR}")
print(f"\nFiles in bronze layer:")
for f in sorted(RAW_DIR.glob("*.csv")):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:40s} {size_mb:>8.1f} MB")

Raw data: /home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/data/raw
Processed data: /home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/data/processed

Files in bronze layer:
  HomeCredit_columns_description.csv            0.0 MB
  POS_CASH_balance.csv                        392.7 MB
  application_test.csv                         26.6 MB
  application_train.csv                       166.1 MB
  bureau.csv                                  170.0 MB
  bureau_balance.csv                          375.6 MB
  credit_card_balance.csv                     424.6 MB
  installments_payments.csv                   723.1 MB
  previous_application.csv                    405.0 MB
  sample_submission.csv                         0.5 MB


## 1. Load All Tables from Bronze Layer

In [2]:
# Load all tables
tables = {}

csv_files = {
    "application_train": "application_train.csv",
    "application_test": "application_test.csv",
    "bureau": "bureau.csv",
    "bureau_balance": "bureau_balance.csv",
    "previous_application": "previous_application.csv",
    "pos_cash_balance": "POS_CASH_balance.csv",
    "credit_card_balance": "credit_card_balance.csv",
    "installments_payments": "installments_payments.csv",
}

for name, filename in csv_files.items():
    filepath = RAW_DIR / filename
    tables[name] = pd.read_csv(filepath)
    print(f"Loaded {name:25s} → {tables[name].shape[0]:>10,} rows × {tables[name].shape[1]:>3} cols")

Loaded application_train         →    307,511 rows × 122 cols


Loaded application_test          →     48,744 rows × 121 cols


Loaded bureau                    →  1,716,428 rows ×  17 cols


Loaded bureau_balance            → 27,299,925 rows ×   3 cols


Loaded previous_application      →  1,670,214 rows ×  37 cols


Loaded pos_cash_balance          → 10,001,358 rows ×   8 cols


Loaded credit_card_balance       →  3,840,312 rows ×  23 cols


Loaded installments_payments     → 13,605,401 rows ×   8 cols


## 2. Schema Inspection

Let's understand the structure, dtypes, and key columns of each table.

In [3]:
def inspect_table(name, df):
    """Print summary stats for a table."""
    print(f"\n{'='*70}")
    print(f"TABLE: {name}")
    print(f"{'='*70}")
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
    
    # Dtype breakdown
    dtype_counts = df.dtypes.value_counts()
    print(f"\nDtype distribution:")
    for dtype, count in dtype_counts.items():
        print(f"  {str(dtype):15s} {count:>3} columns")
    
    # Missing values summary
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(1)
    cols_with_missing = (missing > 0).sum()
    print(f"\nMissing values: {cols_with_missing}/{df.shape[1]} columns have nulls")
    
    if cols_with_missing > 0:
        worst_missing = missing_pct[missing_pct > 0].sort_values(ascending=False).head(5)
        print(f"  Top 5 most missing:")
        for col, pct in worst_missing.items():
            print(f"    {col:40s} {pct:>6.1f}%")
    
    # First few rows
    display(df.head(3))

for name, df in tables.items():
    inspect_table(name, df)


TABLE: application_train
Shape: 307,511 rows × 122 columns


Memory usage: 529.5 MB

Dtype distribution:
  float64          65 columns
  int64            41 columns
  object           16 columns



Missing values: 67/122 columns have nulls
  Top 5 most missing:
    COMMONAREA_AVG                             69.9%
    COMMONAREA_MODE                            69.9%
    COMMONAREA_MEDI                            69.9%
    NONLIVINGAPARTMENTS_AVG                    69.4%
    NONLIVINGAPARTMENTS_MODE                   69.4%


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0



TABLE: application_test
Shape: 48,744 rows × 121 columns


...

Memory usage: 887.5 MB

Dtype distribution:
  float64          15 columns
  int64             7 columns
  object            1 columns



Missing values: 9/23 columns have nulls
  Top 5 most missing:
    AMT_PAYMENT_CURRENT                        20.0%
    AMT_DRAWINGS_ATM_CURRENT                   19.5%
    AMT_DRAWINGS_OTHER_CURRENT                 19.5%
    AMT_DRAWINGS_POS_CURRENT                   19.5%
    CNT_DRAWINGS_ATM_CURRENT                   19.5%


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,...,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,...,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,...,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0



TABLE: installments_payments
Shape: 13,605,401 rows × 8 columns
Memory usage: 870.7 MB

Dtype distribution:
  float64           5 columns
  int64             3 columns

Missing values: 2/8 columns have nulls
  Top 5 most missing:


,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000


## 3. Table Relationships

Understanding how the tables connect via keys.

In [4]:
# Table relationship analysis
print("TABLE RELATIONSHIPS")
print("=" * 70)

# SK_ID_CURR is the main client key across all tables
# SK_ID_BUREAU links bureau → bureau_balance
# SK_ID_PREV links previous_application → POS_CASH, credit_card, installments

relationships = {
    "application_train": {"key": "SK_ID_CURR", "type": "Main table (1 row per application)"},
    "application_test": {"key": "SK_ID_CURR", "type": "Test set (no TARGET)"},
    "bureau": {"key": "SK_ID_CURR → SK_ID_BUREAU", "type": "Many per client"},
    "bureau_balance": {"key": "SK_ID_BUREAU", "type": "Many per bureau record"},
    "previous_application": {"key": "SK_ID_CURR → SK_ID_PREV", "type": "Many per client"},
    "pos_cash_balance": {"key": "SK_ID_PREV → SK_ID_CURR", "type": "Many per prev application"},
    "credit_card_balance": {"key": "SK_ID_PREV → SK_ID_CURR", "type": "Many per prev application"},
    "installments_payments": {"key": "SK_ID_PREV → SK_ID_CURR", "type": "Many per prev application"},
}

for name, info in relationships.items():
    n_rows = len(tables[name])
    print(f"\n{name}")
    print(f"  Key: {info['key']}")
    print(f"  Type: {info['type']}")
    print(f"  Rows: {n_rows:,}")

# Check key overlaps
print("\n" + "=" * 70)
print("KEY COVERAGE")
print("=" * 70)

train_ids = set(tables["application_train"]["SK_ID_CURR"])
test_ids = set(tables["application_test"]["SK_ID_CURR"])
bureau_ids = set(tables["bureau"]["SK_ID_CURR"])
prev_ids = set(tables["previous_application"]["SK_ID_CURR"])

print(f"Train clients:                  {len(train_ids):>10,}")
print(f"Test clients:                   {len(test_ids):>10,}")
print(f"Clients with bureau data:       {len(bureau_ids & train_ids):>10,} / {len(train_ids):,} ({len(bureau_ids & train_ids)/len(train_ids)*100:.1f}%)")
print(f"Clients with previous apps:     {len(prev_ids & train_ids):>10,} / {len(train_ids):,} ({len(prev_ids & train_ids)/len(train_ids)*100:.1f}%)")
print(f"Train/Test overlap:             {len(train_ids & test_ids):>10,} (should be 0)")

TABLE RELATIONSHIPS

application_train
  Key: SK_ID_CURR
  Type: Main table (1 row per application)
  Rows: 307,511

application_test
  Key: SK_ID_CURR
  Type: Test set (no TARGET)
  Rows: 48,744

bureau
  Key: SK_ID_CURR → SK_ID_BUREAU
  Type: Many per client
  Rows: 1,716,428

bureau_balance
  Key: SK_ID_BUREAU
  Type: Many per bureau record
  Rows: 27,299,925

previous_application
  Key: SK_ID_CURR → SK_ID_PREV
  Type: Many per client
  Rows: 1,670,214

pos_cash_balance
  Key: SK_ID_PREV → SK_ID_CURR
  Type: Many per prev application
  Rows: 10,001,358

credit_card_balance
  Key: SK_ID_PREV → SK_ID_CURR
  Type: Many per prev application
  Rows: 3,840,312

installments_payments
  Key: SK_ID_PREV → SK_ID_CURR
  Type: Many per prev application
  Rows: 13,605,401

KEY COVERAGE


Train clients:                     307,511
Test clients:                       48,744
Clients with bureau data:          263,491 / 307,511 (85.7%)
Clients with previous apps:        291,057 / 307,511 (94.6%)
Train/Test overlap:                      0 (should be 0)


## 4. Data Cleaning & Type Optimization

Clean each table: fix dtypes, optimize memory, handle obvious issues.

In [5]:
def optimize_dtypes(df):
    """Downcast numeric types and convert low-cardinality strings to category."""
    result = df.copy()
    
    for col in result.columns:
        col_type = result[col].dtype
        
        if col_type == "int64":
            # Check if it fits in smaller types
            col_min, col_max = result[col].min(), result[col].max()
            if col_min >= np.iinfo(np.int8).min and col_max <= np.iinfo(np.int8).max:
                result[col] = result[col].astype(np.int8)
            elif col_min >= np.iinfo(np.int16).min and col_max <= np.iinfo(np.int16).max:
                result[col] = result[col].astype(np.int16)
            elif col_min >= np.iinfo(np.int32).min and col_max <= np.iinfo(np.int32).max:
                result[col] = result[col].astype(np.int32)
                
        elif col_type == "float64":
            result[col] = result[col].astype(np.float32)
            
        elif col_type == "object":
            n_unique = result[col].nunique()
            if n_unique / len(result) < 0.05:  # Low cardinality → category
                result[col] = result[col].astype("category")
    
    return result

print("Memory optimization per table:\n")
for name in tables:
    before = tables[name].memory_usage(deep=True).sum() / 1e6
    tables[name] = optimize_dtypes(tables[name])
    after = tables[name].memory_usage(deep=True).sum() / 1e6
    reduction = (1 - after / before) * 100
    print(f"  {name:25s} {before:>8.1f} MB → {after:>8.1f} MB  ({reduction:.0f}% reduction)")

Memory optimization per table:



  application_train            529.5 MB →    100.0 MB  (81% reduction)


  application_test              83.6 MB →     15.8 MB  (81% reduction)


  bureau                       495.8 MB →     89.3 MB  (82% reduction)


  bureau_balance              1801.8 MB →    163.8 MB  (91% reduction)


  previous_application        1785.7 MB →    153.7 MB  (91% reduction)


  pos_cash_balance            1112.5 MB →    220.0 MB  (80% reduction)


  credit_card_balance          887.5 MB →    307.2 MB  (65% reduction)


  installments_payments        870.7 MB →    408.2 MB  (53% reduction)


### 4.1 Application Table Cleaning

The main application table needs special attention — fix anomalous values flagged in the dataset documentation.

In [6]:
app_train = tables["application_train"]
app_test = tables["application_test"]

# --- Fix known anomalies ---

# DAYS_BIRTH is negative (days before application). Convert to positive age in years
for df in [app_train, app_test]:
    df["AGE_YEARS"] = (-df["DAYS_BIRTH"] / 365.25).astype(np.float32)

print("Age distribution (years):")
print(app_train["AGE_YEARS"].describe().round(1))

# DAYS_EMPLOYED: 365243 is a sentinel for unemployed/retired
print(f"\nDAYS_EMPLOYED anomaly (365243): {(app_train['DAYS_EMPLOYED'] == 365243).sum():,} rows ({(app_train['DAYS_EMPLOYED'] == 365243).mean()*100:.1f}%)")

for df in [app_train, app_test]:
    df["DAYS_EMPLOYED_ANOMALY"] = (df["DAYS_EMPLOYED"] == 365243).astype(np.int8)
    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

# XNA values in some categorical columns (treated as missing)
for df in [app_train, app_test]:
    for col in df.select_dtypes(include=["category", "object"]).columns:
        if hasattr(df[col], 'cat'):
            if "XNA" in df[col].cat.categories:
                df[col] = df[col].replace("XNA", np.nan)
        else:
            df[col] = df[col].replace("XNA", np.nan)

print("\nCleaned anomalies: DAYS_EMPLOYED sentinel → NaN + flag, XNA → NaN, added AGE_YEARS")

# Update references
tables["application_train"] = app_train
tables["application_test"] = app_test

Age distribution (years):
count    307511.0
mean         43.9
std          11.9
min          20.5
25%          34.0
50%          43.1
75%          53.9
max          69.1
Name: AGE_YEARS, dtype: float64

DAYS_EMPLOYED anomaly (365243): 55,374 rows (18.0%)

Cleaned anomalies: DAYS_EMPLOYED sentinel → NaN + flag, XNA → NaN, added AGE_YEARS


### 4.2 Target Variable Quick Look

In [7]:
target = tables["application_train"]["TARGET"]
print("Target distribution:")
print(target.value_counts())
print(f"\nDefault rate: {target.mean()*100:.2f}%")
print(f"Imbalance ratio: 1:{int((1-target.mean())/target.mean())} (non-default : default)")
print("\n→ Significant class imbalance — will need stratified splits and appropriate metrics (AUC-ROC, PR-AUC)")

Target distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64

Default rate: 8.07%
Imbalance ratio: 1:11 (non-default : default)

→ Significant class imbalance — will need stratified splits and appropriate metrics (AUC-ROC, PR-AUC)


## 5. Save to Silver Layer (Parquet)

Convert all cleaned tables from CSV (bronze) to Parquet (silver). Parquet provides:
- **Columnar storage** — much faster for analytical queries
- **Built-in compression** — typically 3-5x smaller than CSV
- **Schema preservation** — dtypes are stored in the file metadata

In [8]:
print("Saving silver layer (Parquet):\n")

for name, df in tables.items():
    output_path = PROCESSED_DIR / f"{name}.parquet"
    df.to_parquet(output_path, engine="pyarrow", index=False)
    
    csv_size = (RAW_DIR / csv_files[name]).stat().st_size / 1e6
    parquet_size = output_path.stat().st_size / 1e6
    compression = (1 - parquet_size / csv_size) * 100
    
    print(f"  {name:25s} CSV: {csv_size:>7.1f} MB → Parquet: {parquet_size:>7.1f} MB  ({compression:.0f}% compression)")

print("\nSilver layer complete!")

Saving silver layer (Parquet):



  application_train         CSV:   166.1 MB → Parquet:    22.2 MB  (87% compression)


  application_test          CSV:    26.6 MB → Parquet:     4.0 MB  (85% compression)


  bureau                    CSV:   170.0 MB → Parquet:    34.7 MB  (80% compression)


  bureau_balance            CSV:   375.6 MB → Parquet:    16.6 MB  (96% compression)


  previous_application      CSV:   405.0 MB → Parquet:    57.3 MB  (86% compression)


  pos_cash_balance          CSV:   392.7 MB → Parquet:   105.4 MB  (73% compression)


  credit_card_balance       CSV:   424.6 MB → Parquet:   101.5 MB  (76% compression)


  installments_payments     CSV:   723.1 MB → Parquet:   227.2 MB  (69% compression)

Silver layer complete!


## 6. Verification

Read back from silver layer to verify integrity.

In [9]:
print("Verification — reading back from silver layer:\n")

all_ok = True
for name in tables:
    parquet_path = PROCESSED_DIR / f"{name}.parquet"
    df_verify = pd.read_parquet(parquet_path)
    
    rows_match = df_verify.shape[0] == tables[name].shape[0]
    cols_match = df_verify.shape[1] == tables[name].shape[1]
    status = "OK" if (rows_match and cols_match) else "MISMATCH"
    
    if not (rows_match and cols_match):
        all_ok = False
    
    print(f"  {name:25s} {status:>8s}  ({df_verify.shape[0]:>10,} rows × {df_verify.shape[1]:>3} cols)")

print(f"\n{'All tables verified successfully!' if all_ok else 'ERRORS DETECTED — check above'}")
print(f"\nSilver layer files:")
for f in sorted(PROCESSED_DIR.glob("*.parquet")):
    print(f"  {f.name:40s} {f.stat().st_size / 1e6:>7.1f} MB")

Verification — reading back from silver layer:

  application_train               OK  (   307,511 rows × 124 cols)


  application_test                OK  (    48,744 rows × 123 cols)
  bureau                          OK  ( 1,716,428 rows ×  17 cols)


  bureau_balance                  OK  (27,299,925 rows ×   3 cols)


  previous_application            OK  ( 1,670,214 rows ×  37 cols)


  pos_cash_balance                OK  (10,001,358 rows ×   8 cols)


  credit_card_balance             OK  ( 3,840,312 rows ×  23 cols)


  installments_payments           OK  (13,605,401 rows ×   8 cols)

All tables verified successfully!

Silver layer files:
  application_test.parquet                     4.0 MB
  application_train.parquet                   22.2 MB
  bureau.parquet                              34.7 MB
  bureau_balance.parquet                      16.6 MB
  credit_card_balance.parquet                101.5 MB
  installments_payments.parquet              227.2 MB
  pos_cash_balance.parquet                   105.4 MB
  previous_application.parquet                57.3 MB


## Summary

**What we did:**
1. Loaded all 7 CSV tables from the bronze layer (`data/raw/`)
2. Inspected schemas, row counts, dtypes, and missing value patterns
3. Mapped table relationships (SK_ID_CURR, SK_ID_BUREAU, SK_ID_PREV)
4. Cleaned data: optimized dtypes, fixed DAYS_EMPLOYED sentinel, removed XNA values, derived AGE_YEARS
5. Saved cleaned tables as Parquet to the silver layer (`data/processed/`)
6. Verified integrity of silver layer files

**Next:** NB02 — Exploratory Data Analysis (EDA)